# 08 Partitioning and Parallelism

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Control how many partitions a DataFrame has at each step (file reads, shuffles, explicit <code>repartition</code> / <code>coalesce</code>), size them sensibly, detect uneven (skewed) partitions, and connect partitions, cores and tasks into one mental model.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Partitions are the unit of parallelism. Too few, and most of the cluster is idle and big tasks run out of memory. Too many, and scheduling overhead and tiny files dominate. Uneven partitions leave one task running long after the rest. "When would you use repartition vs coalesce?" and "how many partitions should a job have?" are standard interview questions.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Where partition counts come from</b><br>
| Step | Number of partitions decided by |<br>|---|---|<br>| Reading files | File sizes and <code>spark.sql.files.maxPartitionBytes</code> (128 MB): big files are split, small files are packed together (<code>openCostInBytes</code>) |<br>| <code>spark.range</code>, <code>createDataFrame</code> | The default parallelism, or the number you pass |<br>| After a shuffle | <code>spark.sql.shuffle.partitions</code> (200), then AQE coalescing |<br>| <code>repartition(n)</code> / <code>repartition(col)</code> | Exactly n (or the shuffle setting), with a shuffle |<br>| <code>coalesce(n)</code> | At most n, by merging, with no shuffle |<br>| Narrow transformations | Unchanged |<br><br>
<b>Rules of thumb</b>
<ul><li>Aim for partitions of about <b>100–200 MB</b> of data during processing</li><li>Have at least <b>2–3× as many tasks as cores</b>, so work is spread and stragglers matter less</li><li>Look at partition <b>sizes</b>, not just counts: one huge partition is skew</li></ul>
</div>

```
cores = 16, partitions = 4      -> 12 cores idle
cores = 16, partitions = 48     -> 3 waves, good balance
cores = 16, partitions = 50,000 -> overhead: tiny tasks, slow scheduling, tiny files on write
```

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A Silver job reads 10,000 small JSON files (50 KB each) and writes 10,000 tiny Parquet files, and the next job takes 20 minutes just to list and open them. Another job reads one 40 GB gzip file. Gzip isn't splittable, so one task processes all 40 GB while 63 cores wait. The fixes are the same idea in both cases: get the partition count and sizes right, by compacting files, repartitioning after the read, and using splittable formats.
</div>

## Syntax

```python
df.repartition(64)                    # exactly 64, round robin (shuffle)
df.repartition("customer_id")         # hash by key, spark.sql.shuffle.partitions partitions (shuffle)
df.repartition(64, "customer_id")     # hash by key into 64 (shuffle)
df.repartitionByRange(64, "date")     # range partitions, sorted boundaries (shuffle + sampling)
df.coalesce(8)                        # merge down to at most 8 (no shuffle)

spark.conf.set("spark.sql.files.maxPartitionBytes", "64MB")   # smaller read splits
```

## Setup

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Defines a helper that reports partition count and sizes, and creates test files: one larger file and many small ones.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The <code>BASE</code> path, then confirmation that both datasets were written.
</div>

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.spark_basics")
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.spark_basics.raw_files")
BASE = "/Volumes/workspace/spark_basics/raw_files/lesson02_08"

In [0]:
from pyspark.sql import functions as F

def partition_report(df, label=""):
    rows = df.groupBy(F.spark_partition_id().alias("p")).count().collect()
    counts = sorted(r["count"] for r in rows)
    print(f"{label:<32} partitions with data: {len(counts):>4} | min rows: {counts[0]:>8} | max rows: {counts[-1]:>8}")

data = spark.range(2_000_000).select("id", (F.col("id") % 100).alias("k"), F.rand(seed=1).alias("v"))
data.coalesce(1).write.mode("overwrite").parquet(f"{BASE}/one_file")
data.coalesce(1).write.mode("overwrite").option("header", True).csv(f"{BASE}/one_file_csv")
data.repartition(200).write.mode("overwrite").parquet(f"{BASE}/many_files")
print("written")

## 1. Partitions when reading files

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Reads the 200-file and single-file Parquet datasets and reports their partition counts. Then it reads a single CSV file with the default and with a much smaller <code>maxPartitionBytes</code>.<br><br>
<b>Why it matters</b><br>Read parallelism comes from the files. Many small files are <b>packed</b> into fewer partitions (each file still costs an open). A large splittable file is <b>split</b> into chunks of <code>maxPartitionBytes</code>. Lowering that setting creates more, smaller read tasks.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
<code>spark.sql.files.maxPartitionBytes</code> is on the list of settings you can change on serverless. If it is rejected in your workspace, the cell prints a note and keeps going.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The 200 small files are read as far fewer than 200 partitions (they are packed together). The single Parquet file is 1 partition: Parquet can only be split at <b>row-group</b> boundaries, and this small file has one row group. The single CSV file is split by bytes, so a 1 MB split size turns it into many partitions.
</div>

In [0]:
partition_report(spark.read.parquet(f"{BASE}/many_files"), "200 small files")
partition_report(spark.read.parquet(f"{BASE}/one_file"), "1 Parquet file (1 row group)")

csv_schema = "id BIGINT, k BIGINT, v DOUBLE"
partition_report(spark.read.csv(f"{BASE}/one_file_csv", header=True, schema=csv_schema), "1 CSV file, default split size")

original = spark.conf.get("spark.sql.files.maxPartitionBytes")
try:
    spark.conf.set("spark.sql.files.maxPartitionBytes", str(1024 * 1024))     # 1 MB, just for the demo
    partition_report(spark.read.csv(f"{BASE}/one_file_csv", header=True, schema=csv_schema), "1 CSV file, 1 MB split size")
except Exception as e:
    print("maxPartitionBytes can't be changed here:", type(e).__name__)
finally:
    spark.conf.set("spark.sql.files.maxPartitionBytes", original)

## 2. repartition(n), repartition(col), coalesce(n)

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Compares the partition counts and sizes produced by each method.<br><br>
<b>Why it matters</b>
<ul><li><code>repartition(n)</code>: round robin, balanced sizes. Use it to increase parallelism or rebalance</li><li><code>repartition(col)</code>: hash by key, so all rows of a key are in one partition. Sizes follow the key distribution, so they can be uneven</li><li><code>coalesce(n)</code>: cheap merge without a shuffle. Sizes follow the original partitions</li></ul>
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>repartition(16)</code>: 16 balanced partitions. <code>repartition("k")</code>: up to 100 partitions holding data (one per key group, depending on the shuffle setting and AQE). <code>coalesce(4)</code>: 4 partitions.
</div>

In [0]:
base_df = spark.read.parquet(f"{BASE}/many_files")
partition_report(base_df.repartition(16), "repartition(16)")
partition_report(base_df.repartition("k"), "repartition('k')")
partition_report(base_df.repartition(10, "k"), "repartition(10, 'k')")
partition_report(base_df.coalesce(4), "coalesce(4)")

## 3. Detecting skew

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates a dataset where one key holds most of the rows, hash-partitions by that key, and shows the partition sizes.<br><br>
<b>Why it matters</b><br>With skew, the partition count looks fine but one partition is huge, so one task runs much longer than the others. Partition sizes (or the task duration distribution in the Spark UI or query profile) reveal it. Fixes, such as AQE skew join handling, salting and broadcast, are in lesson 13 and <code>05_optimization</code>.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
For <code>repartition(8, "country")</code>, one partition holds about 80% of the rows: the <code>IN</code> key.
</div>

In [0]:
skewed = spark.range(1_000_000).withColumn(
    "country", F.when(F.col("id") % 10 < 8, "IN").otherwise(F.element_at(F.array(*[F.lit(c) for c in ["UK", "AE", "US", "SG"]]), (F.col("id") % 4 + 1).cast("int")))
)
by_country = skewed.repartition(8, "country")
partition_report(by_country, "repartition(8, 'country')")
by_country.groupBy(F.spark_partition_id().alias("partition")).agg(
    F.count("*").alias("rows"), F.collect_set("country").alias("countries")
).orderBy(F.desc("rows")).show()

## 4. The coalesce(1) trap

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Writes one output file two ways, <code>coalesce(1)</code> and <code>repartition(1)</code>, after an expensive step, and compares the times.<br><br>
<b>Why it matters</b><br><code>coalesce</code> doesn't shuffle, so Spark can merge the <b>upstream</b> work into a single task too: the expensive computation then runs on one core. <code>repartition(1)</code> adds a shuffle, so the expensive part still runs in parallel and only the final write is single-threaded.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The <code>repartition(1)</code> version is usually faster when the upstream work is heavy. Both produce one file.
</div>

In [0]:
import time

# Heavy per-row CPU work: hash the value several times
h = F.col("id").cast("string")
for _ in range(6):
    h = F.sha2(h, 256)
expensive = spark.range(0, 3_000_000, 1, numPartitions=8).withColumn("x", h)

for label, df in [("coalesce(1)", expensive.coalesce(1)), ("repartition(1)", expensive.repartition(1))]:
    start = time.time()
    df.write.mode("overwrite").parquet(f"{BASE}/single_{label[:5]}")
    print(f"{label:<15} {time.time() - start:.2f}s")

## 5. Sizing partitions for the data

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Estimates a good partition count from the data size, using the 128 MB rule, and compares it with the number of cores.<br><br>
<b>Why it matters</b><br>A simple, defensible way to choose: total data size divided by the target partition size, then rounded up to a multiple of the cores. On serverless the platform does much of this for you (AQE, auto-sized shuffles), but you still choose partitions when you write files.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A small table of data sizes and suggested partition counts.
</div>

In [0]:
import math

def suggested_partitions(total_mb, cores, target_mb=128):
    by_size = math.ceil(total_mb / target_mb)
    return max(cores * 2, math.ceil(by_size / cores) * cores)

for total_mb, cores in [(500, 8), (10_000, 32), (500_000, 64), (2_000_000, 128)]:
    print(f"{total_mb:>9,} MB on {cores:>3} cores -> {suggested_partitions(total_mb, cores):>6} partitions")

## Under the hood

```
READ        files ──▶ FilePartitions (pack small files, split large splittable files at ~maxPartitionBytes)
                          │  one task each
NARROW      same partitions (no change)
SHUFFLE     map side hashes/round-robins rows ──▶ spark.sql.shuffle.partitions buckets
                          │  AQE: coalesce small buckets, split skewed ones
WRITE       one file per (task × output folder)
```

<span style="background:#1E9E5A;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Narrow</span> `coalesce` merges partitions without moving rows across the network.

<span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Wide</span> `repartition` and `repartitionByRange` shuffle every row.

Non-splittable inputs (gzip CSV/JSON, `multiLine` JSON) are always read as **one task per file**, whatever `maxPartitionBytes` says. Parquet and Delta are splittable by row group.

In [0]:
base_df.repartitionByRange(4, "id").explain()

In [0]:
# Optional cleanup
dbutils.fs.rm(BASE, True)

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: <code>maxPartitionBytes</code> doesn't split my Parquet file</b><br>
Parquet is split at row-group boundaries. A file with one large row group is one task.<br><br>
<b>⛔ Problem: cluster mostly idle, few tasks</b><br>Too few partitions (a single gzip file, <code>coalesce(1)</code> upstream, a tiny <code>spark.range</code>). Repartition after the read, or split the input.<br><br>
<b>⛔ Problem: thousands of tiny tasks and files</b><br>Too many small input files or too many partitions at write time. Compact files, <code>coalesce</code> before writing, or use Delta <code>OPTIMIZE</code>.<br><br>
<b>⛔ Problem: one task takes far longer than the rest</b><br>Skewed partitions. Check partition sizes. See lesson 13 and <code>05_optimization/02_data_skew</code>.<br><br>
**⛔ Problem: <code>coalesce(1)</code> makes the job very slow**<br>The upstream work collapsed into one task. Use <code>repartition(1)</code>, or avoid a single output file.<br><br>
**⛔ Problem: <code>repartition(col)</code> creates many empty or tiny partitions**<br>The key has few distinct values. Use <code>repartition(n, col)</code> with a sensible n.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. How are partitions, cores and tasks related?</b><br>
Each partition is processed by one task, and each task runs on one core. The number of partitions sets how many tasks a stage has, and the number of cores sets how many run at once.<br><br>

<b>🎤 2. What is the difference between <code>repartition</code> and <code>coalesce</code>?</b><br>
<code>repartition</code> does a full shuffle and produces evenly sized partitions, and it can increase or decrease the count. <code>coalesce</code> only decreases the count by merging existing partitions without a shuffle, which is cheaper but can leave uneven partitions and can pull upstream work into fewer tasks.<br><br>

<b>🎤 3. How does Spark decide the number of partitions when reading files?</b><br>
It splits large splittable files into chunks of <code>spark.sql.files.maxPartitionBytes</code> (128 MB by default) and packs small files together, counting an open cost per file. Non-splittable files such as gzip are read whole by one task.<br><br>

<b>🎤 4. How many partitions should a job have?</b><br>
Enough to keep partitions around 100–200 MB and to have at least 2–3 times as many tasks as cores. Then I check sizes for skew. With AQE, Spark coalesces shuffle partitions automatically.<br><br>

<b>🎤 5. When would you use <code>repartition</code> by a column?</b><br>
To co-locate rows with the same key before several operations on that key, or before <code>partitionBy</code> writes, so each output folder gets a small number of files.<br><br>

<b>🎤 6. What does <code>repartitionByRange</code> do?</b><br>
It samples the data to choose boundaries and puts rows into partitions by ranges of the column, so each partition holds a contiguous, sorted range. It's useful before writing data that will be filtered by that column.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. A job reads a single 50 GB <code>.csv.gz</code> file and only one task runs during the read. Why?</b><br>
A. CSV files can never be read in parallel<br>
B. Gzip-compressed files are not splittable, so each file is read by one task<br>
C. The shuffle partition setting is too low<br>
D. Photon is disabled<br>
<details><summary><b>Show answer</b></summary><b>B.</b> Splitting the input into many files, or using a splittable format, fixes it.</details><br><br>

<b>Q2. A team writes a 20 GB DataFrame with 4,000 partitions and gets 4,000 tiny files. What is the simplest fix at write time?</b><br>
A. <code>coalesce</code> to a sensible number of partitions before writing<br>
B. Increase <code>spark.sql.shuffle.partitions</code><br>
C. Cache the DataFrame<br>
D. Use <code>orderBy</code> before writing<br>
<details><summary><b>Show answer</b></summary><b>A.</b> On Delta tables, <code>OPTIMIZE</code> or auto compaction also fixes small files after the fact.</details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Read <code>many_files</code> and report its partitions. Repartition it to exactly 12 balanced partitions and report again</li><li>Using the skewed data, show the rows per partition for <code>repartition(8)</code> (round robin). Is it skewed?</li><li>Write <code>data</code> partitioned by <code>k</code> (100 folders) so that each folder has exactly one file</li><li>In one sentence: why can <code>coalesce(1)</code> be slower than <code>repartition(1)</code>?</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
import os

# 1. Balanced partitions
partition_report(base_df, "many_files as read")
partition_report(base_df.repartition(12), "repartition(12)")

# 2. Round robin ignores keys, so it's balanced even for skewed data
partition_report(skewed.repartition(8), "skewed, repartition(8)")

# 3. One file per folder: repartition by the partition column before partitionBy
data.repartition("k").write.mode("overwrite").partitionBy("k").parquet(f"{BASE}/by_k")
files_per_folder = [len([f for f in fs if f.endswith(".parquet")]) for _, _, fs in os.walk(f"{BASE}/by_k") if any(f.endswith(".parquet") for f in fs)]
print("folders:", len(files_per_folder), "| max files in a folder:", max(files_per_folder))

# 4. coalesce doesn't shuffle, so Spark may run the upstream work in the single remaining task;
#    repartition(1) keeps the upstream work parallel and only funnels the result into one partition.

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>One partition = one task = one core at a time</li><li>File reads: large splittable files are split (<code>maxPartitionBytes</code>), small files are packed, gzip is one task per file</li><li>Shuffles produce <code>spark.sql.shuffle.partitions</code> partitions, and AQE coalesces them</li><li><code>repartition</code> = shuffle, balanced. <code>coalesce</code> = merge, no shuffle, possibly uneven</li><li>Target ~100–200 MB per partition and 2–3× tasks per core. Check sizes for skew</li><li><code>coalesce(1)</code> can serialize upstream work. Prefer <code>repartition(1)</code> when that work is heavy</li></ul>
</div>

| Task | Code |
|---|---|
| Rows per partition | `df.groupBy(F.spark_partition_id()).count()` |
| Exact count, balanced | `df.repartition(n)` |
| By key | `df.repartition(n, "key")` |
| By range | `df.repartitionByRange(n, "col")` |
| Reduce cheaply | `df.coalesce(n)` |
| Read split size | `spark.conf.set("spark.sql.files.maxPartitionBytes", "64MB")` |

## Git commit

```
git add 02_spark_internals/08_partitioning_and_parallelism.ipynb
git commit -m "add 02_08 partitioning and parallelism notebook"
```